# Biểu đồ EDA - Fraud Detection Dataset

Bộ biểu đồ khám phá dữ liệu cho dataset giao dịch 5,000,000 dòng (45 cột).

**Các phần:**
1. Giao dịch diễn ra vào thời điểm nào
2. Sender persona nào bị fraud nhiều nhất
3. Top IP address bị nhiều fraud
4. So sánh Fraud vs Non-Fraud theo transaction_type / payment_channel / device_used
5. Top merchant category
6. Phân bố số tiền giao dịch
7. Tỉ lệ Fraud vs Non-Fraud
8. Location nào diễn ra nhiều fraud nhất

> Đảm bảo `df` (DataFrame) đã được load trước khi chạy các cell bên dưới.

## 0. Cài đặt & config style

In [ ]:
# ==============================================================
# GLOBAL STYLE CONFIG (paste your existing config here if not already run)
# ==============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

PALETTE_FRAUD    = "#FF4C4C"   # red  - fraudulent
PALETTE_LEGIT    = "#4CAEFF"   # blue - legitimate
PALETTE_NEUTRAL  = "#6C63FF"   # purple - single distribution
PALETTE_MERCHANT = "#00C49A"   # teal  - merchant
PALETTE_BG       = "#F8F9FC"   # near-white canvas
PALETTE_CARD     = "#FFFFFF"
PALETTE_TEXT     = "#1A1D2E"
PALETTE_GRID     = "#E2E6F0"

sns.set_theme(style="white", font="DejaVu Sans")

BASE_RC = {
    "figure.facecolor":    PALETTE_BG,
    "axes.facecolor":      PALETTE_CARD,
    "axes.edgecolor":      PALETTE_GRID,
    "axes.labelcolor":     PALETTE_TEXT,
    "axes.labelsize":      13,
    "axes.labelpad":       8,
    "axes.titlesize":      15,
    "axes.titlepad":       12,
    "axes.titleweight":    "bold",
    "axes.spines.top":     False,
    "axes.spines.right":   False,
    "axes.grid":           True,
    "grid.color":          PALETTE_GRID,
    "grid.linewidth":      0.8,
    "grid.alpha":          0.7,
    "xtick.color":         PALETTE_TEXT,
    "ytick.color":         PALETTE_TEXT,
    "xtick.labelsize":     11,
    "ytick.labelsize":     11,
    "figure.dpi":          150,
    "savefig.dpi":         150,
    "savefig.bbox":        "tight",
    "savefig.facecolor":   PALETTE_BG,
}
plt.rcParams.update(BASE_RC)

FRAUD_LABELS   = {0: "Non-Fraud", 1: "Fraud"}
FRAUD_COLORS   = {0: PALETTE_LEGIT, 1: PALETTE_FRAUD}
LEGEND_HANDLES = [
    mpatches.Patch(color=PALETTE_LEGIT, label="\u2714 Non-Fraud"),
    mpatches.Patch(color=PALETTE_FRAUD, label="\u2718 Fraud"),
]

TOP_N = 15  # how many top categories to show in "top N" charts


In [ ]:
def annotate_bars(ax, fmt="{:,.0f}", fontsize=9):
    """Write the value on top of each bar (works for both bar() and barh())."""
    for p in ax.patches:
        w, h = p.get_width(), p.get_height()
        if h != 0 and abs(h) >= abs(w):
            # vertical bar
            val = h
            if pd.isna(val) or val == 0:
                continue
            ax.annotate(fmt.format(val),
                        (p.get_x() + p.get_width() / 2, val),
                        ha="center", va="bottom", fontsize=fontsize,
                        color=PALETTE_TEXT, xytext=(0, 3), textcoords="offset points")
        else:
            # horizontal bar
            val = w
            if pd.isna(val) or val == 0:
                continue
            ax.annotate(fmt.format(val),
                        (val, p.get_y() + p.get_height() / 2),
                        ha="left", va="center", fontsize=fontsize,
                        color=PALETTE_TEXT, xytext=(4, 0), textcoords="offset points")


## Tổng quan các cột
Số lượng non-null, số lượng null, dtype và số giá trị duy nhất của từng cột - kiểm tra nhanh dữ liệu trước khi vẽ biểu đồ.

In [ ]:
# Tổng quan các cột: số lượng giá trị non-null, dtype và số lượng giá trị duy nhất (với cột phân loại)
overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "so_luong_non_null": df.notna().sum(),
    "so_luong_null": df.isna().sum(),
    "so_gia_tri_duy_nhat": df.nunique(),
})
overview.index.name = "cot"
print(f"Tong so dong: {len(df):,} | Tong so cot: {df.shape[1]}")
overview


## 1. Giao dịch diễn ra vào thời điểm nào?
Phân bố số lượng giao dịch theo giờ trong ngày (`txn_hour`), và so sánh giữa ngày thường và cuối tuần (`is_weekend`).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# (a) Number of transactions per hour of day
hour_counts = df["txn_hour"].value_counts().sort_index()
axes[0].bar(hour_counts.index, hour_counts.values, color=PALETTE_NEUTRAL, edgecolor="white")
annotate_bars(axes[0], fmt="{:,.0f}")
axes[0].set_title("Number of transactions by hour of day")
axes[0].set_xlabel("Hour (0-23)")
axes[0].set_ylabel("Transaction count")
axes[0].set_xticks(range(0, 24))

# (b) Hourly pattern: weekday vs weekend
weekend_map = df.groupby(["txn_hour", "is_weekend"]).size().unstack(fill_value=0)
weekend_map.columns = ["Weekday", "Weekend"]
weekend_map.plot(kind="line", marker="o", ax=axes[1],
                  color=[PALETTE_LEGIT, PALETTE_FRAUD], linewidth=2)
axes[1].set_title("Transactions by hour: Weekday vs Weekend")
axes[1].set_xlabel("Hour (0-23)")
axes[1].set_ylabel("Transaction count")
axes[1].legend(title=None, frameon=False)

plt.tight_layout()
plt.show()

print(f"Total transactions: {len(df):,}")
print(hour_counts.rename("count").to_string())


## 2. Sender persona nào bị fraud nhiều nhất?
Trái: số lượng giao dịch fraud theo `sender_persona`. Phải: tỉ lệ fraud (%) theo từng persona - công bằng hơn vì mỗi persona có volume khác nhau.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

persona_fraud_count = (
    df[df["is_fraud"] == 1]["sender_persona"]
    .value_counts()
    .sort_values(ascending=False)
)
axes[0].barh(persona_fraud_count.index[::-1], persona_fraud_count.values[::-1],
             color=PALETTE_FRAUD, edgecolor="white")
annotate_bars(axes[0], fmt="{:,.0f}")
axes[0].set_title("Fraud transaction count by sender_persona")
axes[0].set_xlabel("Fraud transaction count")

persona_rate = df.groupby("sender_persona")["is_fraud"].mean().sort_values(ascending=False) * 100
axes[1].barh(persona_rate.index[::-1], persona_rate.values[::-1],
             color=PALETTE_NEUTRAL, edgecolor="white")
annotate_bars(axes[1], fmt="{:.2f}%")
axes[1].set_title("Fraud rate (%) by sender_persona")
axes[1].set_xlabel("Fraud rate (%)")

plt.tight_layout()
plt.show()

print("Fraud count by persona:")
print(persona_fraud_count.to_string())
print("\nFraud rate (%) by persona:")
print(persona_rate.round(3).to_string())


## 3. Top những IP address bị nhiều fraud nhất
Top N (mặc định `TOP_N`) giá trị `ip_address` được xếp hạng theo số lượng giao dịch fraud.

In [ ]:
ip_fraud_count = (
    df[df["is_fraud"] == 1]["ip_address"]
    .value_counts()
    .head(TOP_N)
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(9, max(5, TOP_N * 0.4)))
ax.barh(ip_fraud_count.index.astype(str), ip_fraud_count.values,
        color=PALETTE_FRAUD, edgecolor="white")
annotate_bars(ax, fmt="{:,.0f}")
ax.set_title(f"Top {TOP_N} IP addresses with the most fraud")
ax.set_xlabel("Fraud transaction count")
plt.tight_layout()
plt.show()

print(ip_fraud_count.sort_values(ascending=False).to_string())


## 4a. So sánh Fraud vs Non-Fraud theo Transaction Type
So sánh giao dịch fraud và non-fraud theo từng `transaction_type` (số lượng tuyệt đối và tỉ lệ fraud %).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# (a) Absolute count: Fraud vs Non-Fraud per category
ct_count = pd.crosstab(df["transaction_type"], df["is_fraud"])
ct_count.columns = [FRAUD_LABELS[c] for c in ct_count.columns]
ct_count = ct_count.sort_values("Fraud", ascending=False)
ct_count.plot(kind="bar", ax=axes[0],
              color=[PALETTE_LEGIT, PALETTE_FRAUD], edgecolor="white", width=0.8)
axes[0].set_title(f"Transaction count by transaction_type")
axes[0].set_xlabel("transaction_type")
axes[0].set_ylabel("Transaction count")
axes[0].tick_params(axis="x", rotation=40)
axes[0].legend(frameon=False)
annotate_bars(axes[0], fmt="{:,.0f}", fontsize=8)

# (b) Fraud rate (%) normalized within each category
ct_pct = pd.crosstab(df["transaction_type"], df["is_fraud"], normalize="index") * 100
ct_pct.columns = [FRAUD_LABELS[c] for c in ct_pct.columns]
ct_pct = ct_pct.sort_values("Fraud", ascending=False)
ct_pct[["Fraud"]].plot(kind="bar", ax=axes[1], color=PALETTE_FRAUD,
                        edgecolor="white", width=0.6, legend=False)
axes[1].set_title(f"Fraud rate (%) by transaction_type")
axes[1].set_xlabel("transaction_type")
axes[1].set_ylabel("Fraud rate (%)")
axes[1].tick_params(axis="x", rotation=40)
annotate_bars(axes[1], fmt="{:.2f}%", fontsize=8)

plt.tight_layout()
plt.show()

print("Count table:")
print(ct_count.to_string())
print("\nFraud rate (%) table:")
print(ct_pct.round(3).to_string())


## 4b. So sánh Fraud vs Non-Fraud theo Payment Channel
So sánh giao dịch fraud và non-fraud theo từng `payment_channel`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# (a) Absolute count: Fraud vs Non-Fraud per category
ct_count = pd.crosstab(df["payment_channel"], df["is_fraud"])
ct_count.columns = [FRAUD_LABELS[c] for c in ct_count.columns]
ct_count = ct_count.sort_values("Fraud", ascending=False)
ct_count.plot(kind="bar", ax=axes[0],
              color=[PALETTE_LEGIT, PALETTE_FRAUD], edgecolor="white", width=0.8)
axes[0].set_title(f"Transaction count by payment_channel")
axes[0].set_xlabel("payment_channel")
axes[0].set_ylabel("Transaction count")
axes[0].tick_params(axis="x", rotation=40)
axes[0].legend(frameon=False)
annotate_bars(axes[0], fmt="{:,.0f}", fontsize=8)

# (b) Fraud rate (%) normalized within each category
ct_pct = pd.crosstab(df["payment_channel"], df["is_fraud"], normalize="index") * 100
ct_pct.columns = [FRAUD_LABELS[c] for c in ct_pct.columns]
ct_pct = ct_pct.sort_values("Fraud", ascending=False)
ct_pct[["Fraud"]].plot(kind="bar", ax=axes[1], color=PALETTE_FRAUD,
                        edgecolor="white", width=0.6, legend=False)
axes[1].set_title(f"Fraud rate (%) by payment_channel")
axes[1].set_xlabel("payment_channel")
axes[1].set_ylabel("Fraud rate (%)")
axes[1].tick_params(axis="x", rotation=40)
annotate_bars(axes[1], fmt="{:.2f}%", fontsize=8)

plt.tight_layout()
plt.show()

print("Count table:")
print(ct_count.to_string())
print("\nFraud rate (%) table:")
print(ct_pct.round(3).to_string())


## 4c. So sánh Fraud vs Non-Fraud theo Device Used
So sánh giao dịch fraud và non-fraud theo từng `device_used`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# (a) Absolute count: Fraud vs Non-Fraud per category
ct_count = pd.crosstab(df["device_used"], df["is_fraud"])
ct_count.columns = [FRAUD_LABELS[c] for c in ct_count.columns]
ct_count = ct_count.sort_values("Fraud", ascending=False)
ct_count.plot(kind="bar", ax=axes[0],
              color=[PALETTE_LEGIT, PALETTE_FRAUD], edgecolor="white", width=0.8)
axes[0].set_title(f"Transaction count by device_used")
axes[0].set_xlabel("device_used")
axes[0].set_ylabel("Transaction count")
axes[0].tick_params(axis="x", rotation=40)
axes[0].legend(frameon=False)
annotate_bars(axes[0], fmt="{:,.0f}", fontsize=8)

# (b) Fraud rate (%) normalized within each category
ct_pct = pd.crosstab(df["device_used"], df["is_fraud"], normalize="index") * 100
ct_pct.columns = [FRAUD_LABELS[c] for c in ct_pct.columns]
ct_pct = ct_pct.sort_values("Fraud", ascending=False)
ct_pct[["Fraud"]].plot(kind="bar", ax=axes[1], color=PALETTE_FRAUD,
                        edgecolor="white", width=0.6, legend=False)
axes[1].set_title(f"Fraud rate (%) by device_used")
axes[1].set_xlabel("device_used")
axes[1].set_ylabel("Fraud rate (%)")
axes[1].tick_params(axis="x", rotation=40)
annotate_bars(axes[1], fmt="{:.2f}%", fontsize=8)

plt.tight_layout()
plt.show()

print("Count table:")
print(ct_count.to_string())
print("\nFraud rate (%) table:")
print(ct_pct.round(3).to_string())


## 5. Top những merchant category được giao dịch nhiều nhất
Top N giá trị `merchant_category` được xếp hạng theo tổng số giao dịch (bao gồm cả fraud và non-fraud).

In [ ]:
merchant_counts = df["merchant_category"].value_counts().head(TOP_N).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, max(5, TOP_N * 0.4)))
ax.barh(merchant_counts.index.astype(str), merchant_counts.values,
        color=PALETTE_MERCHANT, edgecolor="white")
annotate_bars(ax, fmt="{:,.0f}")
ax.set_title(f"Top {TOP_N} merchant categories by transaction count")
ax.set_xlabel("Transaction count")
plt.tight_layout()
plt.show()

print(merchant_counts.sort_values(ascending=False).to_string())


## 6. Số tiền giao dịch thường thấp hay cao?
Phân bố `amount_ngn` ở cả thang tuyến tính và log, kèm mean/median để xác định độ lệch phải.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(df["amount_ngn"], bins=80, color=PALETTE_NEUTRAL, ax=axes[0], kde=False)
mean_val = df["amount_ngn"].mean()
median_val = df["amount_ngn"].median()
axes[0].axvline(mean_val, color=PALETTE_FRAUD, linestyle="--", linewidth=1.5,
                 label=f"Mean: {mean_val:,.0f}")
axes[0].axvline(median_val, color=PALETTE_MERCHANT, linestyle="--", linewidth=1.5,
                 label=f"Median: {median_val:,.0f}")
axes[0].set_title("Transaction amount distribution (amount_ngn)")
axes[0].set_xlabel("Amount (NGN)")
axes[0].legend(frameon=False)

sns.histplot(df["amount_ngn"], bins=80, color=PALETTE_NEUTRAL, ax=axes[1], kde=False, log_scale=True)
axes[1].set_title("Transaction amount distribution (log scale)")
axes[1].set_xlabel("Amount (NGN, log scale)")

plt.tight_layout()
plt.show()

print(f"Count:  {df['amount_ngn'].count():,}")
print(f"Mean:   {mean_val:,.2f} NGN")
print(f"Median: {median_val:,.2f} NGN")
print(f"Min:    {df['amount_ngn'].min():,.2f} NGN")
print(f"Max:    {df['amount_ngn'].max():,.2f} NGN")
print("-> If Mean >> Median: right-skewed distribution (most transactions are small, a few are very large).")


## 7. Tỉ lệ Fraud so với Non-Fraud
Tỉ lệ tổng thể của `is_fraud` (0 = non-fraud, 1 = fraud) trong toàn bộ dataset - thường là tỉ lệ mất cân bằng rất mạnh.

In [ ]:
counts = df["is_fraud"].value_counts().sort_index()
labels = [FRAUD_LABELS[i] for i in counts.index]
colors = [FRAUD_COLORS[i] for i in counts.index]

fig, ax = plt.subplots(figsize=(6, 6))
wedges, texts, autotexts = ax.pie(
    counts.values, labels=labels, colors=colors, autopct="%1.3f%%",
    startangle=90, pctdistance=0.75,
    wedgeprops=dict(width=0.4, edgecolor="white"),
    textprops=dict(color=PALETTE_TEXT, fontsize=12),
)
ax.set_title("Fraud vs Non-Fraud ratio")
plt.tight_layout()
plt.show()

total = counts.sum()
for i, c in counts.items():
    print(f"{FRAUD_LABELS[i]}: {c:,} transactions ({c/total*100:.4f}%)")


## 8. Location nào diễn ra nhiều fraud nhất?
Trái: top N giá trị `location` xếp hạng theo số lượng fraud tuyệt đối. Phải: top N location xếp hạng theo tỉ lệ fraud (%).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

loc_fraud_count = (
    df[df["is_fraud"] == 1]["location"]
    .value_counts()
    .head(TOP_N)
    .sort_values(ascending=True)
)
axes[0].barh(loc_fraud_count.index.astype(str), loc_fraud_count.values,
             color=PALETTE_FRAUD, edgecolor="white")
annotate_bars(axes[0], fmt="{:,.0f}")
axes[0].set_title(f"Top {TOP_N} locations with the most fraud (count)")
axes[0].set_xlabel("Fraud transaction count")

loc_fraud_rate = (
    df.groupby("location")["is_fraud"].mean().sort_values(ascending=False).head(TOP_N) * 100
).sort_values(ascending=True)
axes[1].barh(loc_fraud_rate.index.astype(str), loc_fraud_rate.values,
             color=PALETTE_NEUTRAL, edgecolor="white")
annotate_bars(axes[1], fmt="{:.2f}%")
axes[1].set_title(f"Top {TOP_N} locations with the highest fraud rate (%)")
axes[1].set_xlabel("Fraud rate (%)")

plt.tight_layout()
plt.show()

print("Fraud count by location:")
print(loc_fraud_count.sort_values(ascending=False).to_string())
print("\nFraud rate (%) by location:")
print(loc_fraud_rate.sort_values(ascending=False).round(3).to_string())
